# 06 — Learning to rank

**What this shows**

- A learned ranker (LambdaMART) is compared with the strongest *single* hand-built signal on Microsoft's MSLR-WEB10K, the standard learning-to-rank benchmark.
- The single feature is chosen on the validation split, never on test, so the baseline is as strong as one feature can honestly be.
- The comparison uses the same ruler as notebooks 01 to 05: `ndcg_at_k` from `ragexp.metrics` and the paired bootstrap over test queries.

## The prediction, written before running

> **LambdaMART beats the best single feature.**
>
> One feature sees one signal. A listwise tree ensemble combines all 136 and optimises
> the ranking metric directly. On LTR's home benchmark that should show.
>
> **Falsified if** the paired-bootstrap 95% CI on nDCG@10 (LambdaMART minus best single
> feature, Fold 1 test queries) includes 0 or lies below it.

This refines the row first written in the README ("LambdaMART wins"), which named no
baseline. The refinement was committed on 2026-10-29, before any model was trained or any
feature was scored. The README row also asked whether a published null was due to its
LLM-generated labels. MSLR-WEB10K ships only numeric features and human grades, with no
query or document text, so nothing here can be relabelled. That question stays open as a
caveat.

## Setup, fixed in advance

- **Data:** MSLR-WEB10K, Fold 1 only: `train.txt`, `vali.txt`, `test.txt`. 136 features,
  relevance grades 0 to 4. Read with `sklearn.datasets.load_svmlight_file(query_id=True)`
  from a local copy outside the repo (the licence does not allow redistribution).
- **LambdaMART:** `xgboost.XGBRanker(objective="rank:ndcg", n_estimators=2000,
  learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
  tree_method="hist", eval_metric="ndcg@10", early_stopping_rounds=100, random_state=0)`.
  Early stopping watches the validation split. No other tuning.
- **Best single feature:** for each of the 136 features, in both directions (higher is
  better, lower is better), rank each validation query's documents by that value and score
  mean nDCG@10. The best of those 272 rankings is the baseline. Ties keep file order.
- **Metric:** nDCG@10 on the test split with `ragexp.metrics.ndcg_at_k`. A query with no
  document graded above 0 is undefined (`None`) and dropped, as in notebook 01.
- **Test:** `paired_bootstrap` with the default seed, queries in file order.